# MolHIV Fine-Tuned Graphormer Attention Interpretability — Figure Generation

This notebook mirrors `generate_figures.ipynb`, but loads a newly fine-tuned OGBG-MolHIV Graphormer checkpoint from Drive/local run outputs and uses OGBG-MolHIV SMILES instead of PCQM4Mv2 molecules.


In [ ]:
# ── Clone repo, mount Drive, install dependencies, restart once ──
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped/unavailable: {exc}')

if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', 'https://github.com/joshgreenwa/Symbolic-and-Structural-Specialisation-in-Graphormers.git', '/content/repo'], check=True)

if '/content/repo' not in sys.path:
    sys.path.insert(0, '/content/repo')

os.chdir('/content/repo')

sentinel = '/content/.molhiv_analysis_deps_installed'
if not os.path.exists(sentinel):
    print('Installing a clean Colab analysis stack, then restarting the kernel once...')
    pkgs = [
        'numpy==1.26.4',
        'scipy==1.11.4',
        'scikit-learn==1.4.2',
        'matplotlib==3.8.4',
        'pandas==2.2.2',
        'ogb==1.3.6',
        'torch==2.2.2',
        'torchaudio==2.2.2',
        'torchvision==0.17.2',
        'transformers==4.40.2',
        'safetensors==0.7.0',
        'tokenizers==0.19.1',
        'rdkit',
    ]
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '-i', 'https://pypi.org/simple', *pkgs],
        check=True,
    )
    open(sentinel, 'w').write('ok\n')
    print('Dependencies installed. Restarting kernel now; rerun this cell once after restart, then run the next cell.')
    os.kill(os.getpid(), 9)
else:
    print('MolHIV analysis dependencies already installed for this runtime.')

print('MolHIV analysis setup complete')


In [ ]:
# ── Imports and MolHIV checkpoint setup ──
import csv
import gzip
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from graph_interp.config import MODEL_ID, MODEL_REVISION, set_seed, apply_plot_defaults
import graph_interp.data as graph_data
import graph_interp.extraction as graph_extraction

# Existing repo functions used by all downstream figure cells.
build_batch_from_smiles = graph_data.build_batch_from_smiles
extract_attention = graph_extraction.extract_attention
extract_attention_with_ablation = graph_extraction.extract_attention_with_ablation
extract_hidden_states = graph_extraction.extract_hidden_states
extract_with_values = graph_extraction.extract_with_values
extract_spatial_bias_profiles = graph_extraction.extract_spatial_bias_profiles
node_only_conditional = graph_extraction.node_only_conditional


def _as_index_list(indices):
    if hasattr(indices, "detach"):
        indices = indices.detach().cpu()
    if hasattr(indices, "numpy"):
        indices = indices.numpy()
    if hasattr(indices, "tolist"):
        indices = indices.tolist()
    return [int(i) for i in indices]


def _read_smiles_mapping(path: Path) -> list[str]:
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames:
            raise ValueError(f"Mapping file has no header: {path}")
        smiles_key = next(
            (key for key in reader.fieldnames if key.lower() in {"smiles", "smi"}),
            None,
        )
        if smiles_key is None:
            raise ValueError(f"No SMILES column found in {path}; fields={reader.fieldnames}")
        return [row[smiles_key] for row in reader]


def _find_molhiv_smiles_mapping(root: str | Path) -> Path:
    root = Path(root)
    candidates = [
        root / "ogbg_molhiv" / "mapping" / "mol.csv.gz",
        root / "ogbg_molhiv" / "mapping" / "mol.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    matches = sorted(root.rglob("mol.csv*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find OGBG-MolHIV mol.csv mapping under {root}")


def _fallback_load_molhiv_smiles_pool(root="dataset", split="train", pool_size=2000):
    from ogb.graphproppred import GraphPropPredDataset

    ds = GraphPropPredDataset(name="ogbg-molhiv", root=root)
    split_idx = ds.get_idx_split()
    if split not in split_idx:
        raise KeyError(f"Unknown MolHIV split {split!r}; available={sorted(split_idx)}")
    smiles = _read_smiles_mapping(_find_molhiv_smiles_mapping(root))
    indices = _as_index_list(split_idx[split])
    return [smiles[i] for i in indices[: min(pool_size, len(indices))]]


load_molhiv_smiles_pool = getattr(
    graph_data, "load_molhiv_smiles_pool", _fallback_load_molhiv_smiles_pool
)
if load_molhiv_smiles_pool is _fallback_load_molhiv_smiles_pool:
    print("Using local fallback load_molhiv_smiles_pool; /content/repo is missing the new helper.")


def _resolve_checkpoint_path(checkpoint_path=None, run_dir=None, ckpt_dir=None):
    if checkpoint_path:
        path = Path(checkpoint_path).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"Checkpoint file not found: {path}")
        return path

    if ckpt_dir is None:
        if run_dir is None:
            for candidate in [
                Path("/content/drive/MyDrive/graphormer_molhiv_runs/official_molhiv_pcqm4mv1_graphormer_base_for_molhiv_seed1"),
                Path("/content/graphormer_molhiv_runs/official_molhiv_pcqm4mv1_graphormer_base_for_molhiv_seed1"),
                Path("/content/drive/MyDrive/graphormer_molhiv_runs/official_molhiv_pcqm4mv1_graphormer_base_seed1"),
                Path("/content/graphormer_molhiv_runs/official_molhiv_pcqm4mv1_graphormer_base_seed1"),
            ]:
                if candidate.exists():
                    run_dir = candidate
                    break
        if run_dir is None:
            raise FileNotFoundError("Provide MOLHIV_CHECKPOINT or MOLHIV_RUN_DIR.")
        ckpt_dir = Path(run_dir).expanduser() / "ckpts"
    else:
        ckpt_dir = Path(ckpt_dir).expanduser()

    for name in ("checkpoint_best_valid_auc.pt", "checkpoint_best.pt", "checkpoint_last.pt"):
        path = ckpt_dir / name
        if path.is_file():
            return path

    ckpts = sorted(
        (p for p in ckpt_dir.glob("checkpoint*.pt") if "pretrained_step0" not in p.name),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if ckpts:
        return ckpts[0]
    raise FileNotFoundError(f"No checkpoint*.pt files found in {ckpt_dir}")


def _fairseq_encoder_state(checkpoint):
    state = checkpoint.get("model", checkpoint)
    encoder_state = {}
    for key, value in state.items():
        if key.startswith("encoder."):
            encoder_state[key[len("encoder."):]] = value
    if not encoder_state:
        raise RuntimeError("No 'encoder.' keys found in checkpoint model state.")
    return encoder_state


def _lookup_nested(obj, path):
    cur = obj
    for key in path:
        if isinstance(cur, dict):
            cur = cur.get(key)
        else:
            cur = getattr(cur, key, None)
        if cur is None:
            return None
    return cur


def _infer_pre_layernorm(checkpoint, checkpoint_path):
    for path in (("cfg", "model", "pre_layernorm"), ("args", "pre_layernorm"), ("extra_state", "pre_layernorm")):
        value = _lookup_nested(checkpoint, path)
        if value is not None:
            return bool(value)
    return "base_for_molhiv" in str(checkpoint_path)


def _set_pre_layernorm(model, enabled):
    if hasattr(model, "config"):
        setattr(model.config, "pre_layernorm", enabled)
    graph_encoder = getattr(model, "graph_encoder", None)
    if graph_encoder is not None:
        for layer in getattr(graph_encoder, "layers", []):
            if hasattr(layer, "pre_layernorm"):
                layer.pre_layernorm = enabled


def _fallback_load_molhiv_model(
    checkpoint_path=None,
    run_dir=None,
    ckpt_dir=None,
    device="cuda",
    base_model_id=MODEL_ID,
    base_revision=MODEL_REVISION,
):
    from transformers import GraphormerForGraphClassification

    if isinstance(device, str):
        device = torch.device(device if torch.cuda.is_available() or device == "cpu" else "cpu")

    checkpoint_path = _resolve_checkpoint_path(checkpoint_path, run_dir, ckpt_dir)
    checkpoint = torch.load(str(checkpoint_path), map_location="cpu")
    encoder_state = _fairseq_encoder_state(checkpoint)
    pre_layernorm = _infer_pre_layernorm(checkpoint, checkpoint_path)

    clf, _ = GraphormerForGraphClassification.from_pretrained(
        base_model_id,
        revision=base_revision,
        use_safetensors=True,
        output_loading_info=True,
    )
    model = clf.encoder
    _set_pre_layernorm(model, pre_layernorm)
    target_state = model.state_dict()

    compatible_state = {}
    skipped = []
    for key, value in encoder_state.items():
        if key in target_state and tuple(target_state[key].shape) == tuple(value.shape):
            compatible_state[key] = value
        else:
            skipped.append(key)

    missing, unexpected = model.load_state_dict(compatible_state, strict=False)
    critical_missing = [
        key for key in missing if key.startswith("graph_encoder.") and "embed_out" not in key
    ]
    if critical_missing:
        raise RuntimeError(f"Missing critical graph encoder keys: {critical_missing[:8]}")
    if unexpected:
        raise RuntimeError(f"Unexpected keys while loading MolHIV checkpoint: {unexpected[:8]}")
    if skipped:
        print(f"Skipped {len(skipped)} non-encoder/shape-mismatched keys; first few: {skipped[:5]}")

    return model.to(device).eval(), device, checkpoint_path


load_molhiv_model = getattr(graph_extraction, "load_molhiv_model", _fallback_load_molhiv_model)
if load_molhiv_model is _fallback_load_molhiv_model:
    print("Using local fallback load_molhiv_model; /content/repo is missing the new helper.")

apply_plot_defaults()
set_seed(42)

# Override these in a small cell before this one if needed.
MOLHIV_RUN_DIR = os.environ.get(
    "MOLHIV_RUN_DIR",
    "/content/drive/MyDrive/graphormer_molhiv_runs/official_molhiv_pcqm4mv1_graphormer_base_for_molhiv_seed1",
)
MOLHIV_CHECKPOINT = os.environ.get("MOLHIV_CHECKPOINT", "")
MOLHIV_DATA_ROOT = os.environ.get("MOLHIV_DATA_ROOT", "dataset")
MOLHIV_SPLIT = os.environ.get("MOLHIV_SPLIT", "train")
MOLHIV_POOL_SIZE = int(os.environ.get("MOLHIV_POOL_SIZE", "2000"))

if MOLHIV_CHECKPOINT:
    model, device, checkpoint_path = load_molhiv_model(checkpoint_path=MOLHIV_CHECKPOINT)
else:
    model, device, checkpoint_path = load_molhiv_model(run_dir=MOLHIV_RUN_DIR)

smiles_pool = load_molhiv_smiles_pool(
    root=MOLHIV_DATA_ROOT, split=MOLHIV_SPLIT, pool_size=MOLHIV_POOL_SIZE,
)
print(f"MolHIV model loaded on {device}: {checkpoint_path}")
print(f"MolHIV SMILES pool split={MOLHIV_SPLIT!r}: {len(smiles_pool)} molecules")


---
## Section A: Baseline Scores and Interpretation (Figures 1–5)

In [ ]:
# ── Section A: Compute baseline scores ──
N = 10          # ← local N for this section
num_perms = 96  # transpositions per graph

from graph_interp.metrics.scores import aggregate_scores

mean_struct, mean_sym, mean_diff, std_struct, std_sym = aggregate_scores(
    model, smiles_pool, device, n_graphs=N, num_perms=num_perms, return_std=True,
)
print(f'Scores computed: {mean_struct.shape}  (N={N})')

In [ ]:
# Fig 1: Symbolic score heatmap
from graph_interp.plots.section_a import fig01_symbolic_score_heatmap
fig = fig01_symbolic_score_heatmap(mean_sym, N=N)
plt.show()

In [ ]:
# Fig 2: Structural score heatmap
from graph_interp.plots.section_a import fig02_structural_score_heatmap
fig = fig02_structural_score_heatmap(mean_struct, N=N)
plt.show()

In [ ]:
# Fig 3: Score distribution coloured by layer (with L1,H24 and L11,H14 annotated)
from graph_interp.plots.section_a import fig03_score_scatter_by_layer
fig = fig03_score_scatter_by_layer(mean_struct, mean_sym, N=N)
plt.show()

In [ ]:
# Fig 4: Score distribution coloured by head index
from graph_interp.plots.section_a import fig04_score_scatter_by_head
fig = fig04_score_scatter_by_head(mean_struct, mean_sym, N=N)
plt.show()

In [ ]:
# ── Section A (cont.): Compute scoring variants for Fig 5 ──
from graph_interp.metrics.variants import aggregate_all_variants

variants, n_variants_used = aggregate_all_variants(
    model, smiles_pool, device, n_graphs=N, num_perms=num_perms,
)
print(f'Variants computed: {n_variants_used}/{N} graphs used')

In [ ]:
# Fig 4b and Fig 5: centered-cosine scatter + scoring variants comparison
from graph_interp.plots.section_a import (
    fig04b_centered_cossim_scatter_by_layer,
    fig05_scoring_variants_comparison,
)

fig = fig04b_centered_cossim_scatter_by_layer(variants, N=n_variants_used)
plt.show()

fig = fig05_scoring_variants_comparison(variants, N=n_variants_used)
plt.show()


---
## Section B: Dot-Product vs Bias Dominance (Figures 6–9)

In [ ]:
# ── Section B: Compute logit spread statistics ──
N_B = 10  # ← local N for this section

from graph_interp.metrics.logit_spread import aggregate_logit_spread

logit_data = aggregate_logit_spread(model, smiles_pool, device, n_graphs=N_B)
print(f'Logit spread computed (N={N_B})')

In [ ]:
# Fig 6: Logit spread across layers (with error bands)
from graph_interp.plots.section_b import fig06_logit_spread
fig = fig06_logit_spread(
    logit_data['dot_std_mean'], logit_data['bias_std_mean'],
    dot_std_err=logit_data.get('dot_std_std'),
    bias_std_err=logit_data.get('bias_std_std'),
    N=N_B,
)
plt.show()

In [ ]:
# Fig 7: Dot/bias variance ratio
from graph_interp.plots.section_b import fig07_variance_ratio
fig = fig07_variance_ratio(logit_data['var_ratio'], N=N_B)
plt.show()

In [ ]:
# Fig 8: Score difference vs log-bias ratio
from graph_interp.plots.section_b import fig08_structural_dom_vs_bias_ratio
fig = fig08_structural_dom_vs_bias_ratio(
    mean_struct, mean_sym, logit_data['log_r_mean'], N=N_B,
)
plt.show()

In [ ]:
# Fig 9: Bias ratio per head across layers
from graph_interp.plots.section_b import fig09_bias_ratio_per_head
fig = fig09_bias_ratio_per_head(logit_data['log_r_mean'], N=N_B)
plt.show()

---
## Section C: Early-Layer Symbolic Specialisation (Figures 10–12)

In [ ]:
# ── Section C: Extract attention for example molecules ──
mol_indices = [0, 5, 80]  # ← configurable molecule indices
# Clamp to available range
mol_indices = [i for i in mol_indices if i < len(smiles_pool)]

example_smiles = [smiles_pool[i] for i in mol_indices]
A_list_per_mol = []

for smi in example_smiles:
    batch, n, dist = build_batch_from_smiles(smi, model, device)
    _, _, A_list = extract_attention(model, batch)
    A_list_per_mol.append(A_list)

print(f'Extracted attention for molecules {mol_indices}')

In [ ]:
# Fig 10: L1,H24 — 3-panel attention visualisation (molecule, overlay, attention matrix)
from graph_interp.plots.section_c import fig10_symbolic_head_attention
figs = fig10_symbolic_head_attention(
    A_list_per_mol, example_smiles,
    layer=1, head=24, mol_indices=list(range(len(example_smiles))),
)
for f in figs:
    plt.show()


In [ ]:
# ── Section C: PCA of head output (A@V) for L1,H24 ──
from graph_interp.chemistry import label_head_focus_expanded

FOCUS_THRESH = 0.75
DIFFUSE_THRESH = 0.35

N_PCA = 500  # ← molecules for PCA
pca_smiles = smiles_pool[:N_PCA]

embeddings_l1h24 = []
labels_l1h24 = []

for smi in pca_smiles:
    try:
        batch, n, dist = build_batch_from_smiles(smi, model, device)
        _, _, A_list_tmp = extract_attention(model, batch)
        _, _, _, AV_list = extract_with_values(model, batch)

        # Pooled head output: mean of (A@V) over node queries for L1,H24
        av = AV_list[1][24, 1:, :].mean(dim=0).cpu().numpy()  # [Hd]
        embeddings_l1h24.append(av)

        # Expanded chemistry focus label from inbound node->node mass
        p_node, _ = node_only_conditional(A_list_tmp[1])
        p_atom = p_node[24].mean(dim=0).cpu().numpy()  # [n]
        labels_l1h24.append(
            label_head_focus_expanded(
                smi, p_atom,
                focus_mass=FOCUS_THRESH,
                diffuse_thresh=DIFFUSE_THRESH,
            )
        )
    except Exception:
        continue

embeddings_l1h24 = np.array(embeddings_l1h24)
print(f'PCA embeddings L1,H24: {embeddings_l1h24.shape}')


In [ ]:
# Fig 11: PCA of A@V head output — L1,H24
from graph_interp.plots.section_c import fig11_pca_head_output
fig = fig11_pca_head_output(embeddings_l1h24, labels_l1h24, layer=1, head=24, N=len(embeddings_l1h24))
plt.show()

In [ ]:
# ── Section C: PCA for a second early-layer head (L0,H0) ──
embeddings_l0h0 = []
labels_l0h0 = []

for smi in pca_smiles:
    try:
        batch, n, dist = build_batch_from_smiles(smi, model, device)
        _, _, A_list_tmp = extract_attention(model, batch)
        _, _, _, AV_list = extract_with_values(model, batch)

        av = AV_list[0][0, 1:, :].mean(dim=0).cpu().numpy()
        embeddings_l0h0.append(av)

        p_node, _ = node_only_conditional(A_list_tmp[0])
        p_atom = p_node[0].mean(dim=0).cpu().numpy()
        labels_l0h0.append(
            label_head_focus_expanded(
                smi, p_atom,
                focus_mass=FOCUS_THRESH,
                diffuse_thresh=DIFFUSE_THRESH,
            )
        )
    except Exception:
        continue

embeddings_l0h0 = np.array(embeddings_l0h0)
print(f'PCA embeddings L0,H0: {embeddings_l0h0.shape}')


In [ ]:
# Fig 12: PCA of A@V head output — L0,H0
from graph_interp.plots.section_c import fig12_pca_head_output_second
fig = fig12_pca_head_output_second(embeddings_l0h0, labels_l0h0, layer=0, head=0, N=len(embeddings_l0h0))
plt.show()

---
## Section D: Structural Specialisation (Figures 13–14)

In [ ]:
# Fig 13: L11,H14 — 3-panel attention visualisation
from graph_interp.plots.section_d import fig13_structural_head_attention
figs = fig13_structural_head_attention(
    A_list_per_mol, example_smiles,
    layer=11, head=14, mol_indices=list(range(len(example_smiles))),
)
for f in figs:
    plt.show()

In [ ]:
# ── Section D: Distance statistics for Fig 14 ──
N_D = 10  # ← local N

from graph_interp.metrics.distance import aggregate_distance_stats
mean_mu, mean_sigma, mean_M = aggregate_distance_stats(
    model, smiles_pool, device, n_graphs=N_D,
)
print(f'Distance stats: {mean_M.shape}  (N={N_D})')

In [ ]:
# Fig 14: Attention mass vs hop distance — L11,H14
from graph_interp.plots.section_d import fig14_attention_mass_vs_hop
fig = fig14_attention_mass_vs_hop(mean_M, layer=11, head=14, N=N_D)
plt.show()

---
## Section E: Bias Routes, Dot Selects Within (Figures 15–19)

In [ ]:
# ── Section E: Compute bias routing metrics ──
N_E = 10  # ← local N

from graph_interp.metrics.bias_routing import aggregate_bias_routing
containment_LH, cossim_results = aggregate_bias_routing(
    model, smiles_pool, device, n_graphs=N_E,
)
print(f'Bias routing computed: {containment_LH.shape}  (N={N_E})')

In [ ]:
# Fig 15–19
from graph_interp.plots.section_e import (
    fig15_containment_heatmap, fig16_structural_score_topk_vs_botk,
    fig17_symbolic_score_topk_vs_botk, fig18_symbolic_gap, fig19_structural_gap,
)

fig = fig15_containment_heatmap(containment_LH, N=N_E); plt.show()
fig = fig16_structural_score_topk_vs_botk(cossim_results, N=N_E); plt.show()
fig = fig17_symbolic_score_topk_vs_botk(cossim_results, N=N_E); plt.show()
fig = fig18_symbolic_gap(cossim_results, N=N_E); plt.show()
fig = fig19_structural_gap(cossim_results, N=N_E); plt.show()

---
## Section F: CLS Token Dynamics (Figures 20–28)

In [ ]:
# ── Section F: Compute CLS metrics (with Welford std for error bars) ──
N_F = 10  # ← local N

from graph_interp.metrics.cls import aggregate_cls_metrics
cls_metrics = aggregate_cls_metrics(model, smiles_pool, device, n_graphs=N_F)
print(f'CLS metrics computed (N={N_F})')

In [ ]:
# Fig 20–28 (CLS heatmaps and line plots with error bars)
from graph_interp.plots.section_f import (
    fig20_cls_structural_heatmap, fig21_cls_centered_heatmaps,
    fig22_cls_entropy_heatmap, fig23_cls_gini_heatmap,
    fig24_cls_entropy_across_layers, fig25_cls_max_attention,
    fig26_cls_vs_node_entropy, fig27_cls_vs_node_dot_spread,
    fig28_cls_similarity_to_mean,
)

fig = fig20_cls_structural_heatmap(cls_metrics, N=N_F); plt.show()
fig = fig21_cls_centered_heatmaps(cls_metrics, N=N_F); plt.show()
fig = fig22_cls_entropy_heatmap(cls_metrics, N=N_F); plt.show()
fig = fig23_cls_gini_heatmap(cls_metrics, N=N_F); plt.show()
fig = fig24_cls_entropy_across_layers(cls_metrics, N=N_F); plt.show()
fig = fig25_cls_max_attention(cls_metrics, N=N_F); plt.show()
fig = fig26_cls_vs_node_entropy(cls_metrics, N=N_F); plt.show()
fig = fig27_cls_vs_node_dot_spread(cls_metrics, N=N_F); plt.show()
fig = fig28_cls_similarity_to_mean(cls_metrics, N=N_F); plt.show()

---
## Section G: Representation Dynamics (Figures 29–32)

In [ ]:
# ── Section G: Compute representation dynamics ──
N_G = 10  # ← local N

from graph_interp.metrics.representation import (
    aggregate_representation_dynamics, aggregate_sublayer_breakdown,
)
mean_sim, min_sim, max_sim, delta_sim = aggregate_representation_dynamics(
    model, smiles_pool, device, n_graphs=N_G,
)
sim_input, sim_attn, sim_ffn = aggregate_sublayer_breakdown(
    model, smiles_pool, device, n_graphs=N_G,
)
print(f'Representation dynamics computed (N={N_G})')

In [ ]:
# Fig 29–32
from graph_interp.plots.section_g import (
    fig29_node_similarity_across_layers, fig30_delta_similarity,
    fig31_sublayer_similarity, fig32_attn_vs_ffn_contribution,
)

fig = fig29_node_similarity_across_layers(mean_sim, min_sim, max_sim, N=N_G); plt.show()
fig = fig30_delta_similarity(delta_sim, N=N_G); plt.show()
fig = fig31_sublayer_similarity(sim_input, sim_attn, sim_ffn, N=N_G); plt.show()
fig = fig32_attn_vs_ffn_contribution(sim_input, sim_attn, sim_ffn, N=N_G); plt.show()

---
## Section H: Head Ablation — L1, H24 (Figures 33–36)

In [ ]:
# ── Section H: Compute ablation effects ──
N_H = 10  # ← local N

ablate_heads = {1: [24]}
mean_tv_L = np.zeros(12)
max_tv_L = np.zeros(12)
effect_LH = np.zeros((12, 32))
gini_per_layer = np.zeros(12)
cls_perturbation = np.zeros(12)
cls_perturbation_sq = np.zeros(12)
n_abl = 0

for smi in smiles_pool[:N_H]:
    try:
        batch, n, dist = build_batch_from_smiles(smi, model, device)
        _, _, A_base = extract_attention(model, batch)
        _, _, A_abl = extract_attention_with_ablation(model, batch, ablate_heads=ablate_heads)

        _, hidden_base, _ = extract_hidden_states(model, batch)
        _, hidden_abl, _ = extract_hidden_states(model, batch, ablate_heads=ablate_heads)

        for l in range(12):
            tv = 0.5 * (A_base[l] - A_abl[l]).abs().sum(dim=-1)  # [H, T]
            tv_per_head = tv.mean(dim=-1).cpu().numpy()  # [H]
            effect_LH[l] += tv_per_head
            mean_tv_L[l] += tv_per_head.mean()
            max_tv_L[l] += tv_per_head.max()

            delta_h = (hidden_base[l] - hidden_abl[l]).float()
            node_pert = delta_h[1:].norm(dim=-1).cpu().numpy()
            if len(node_pert) > 1:
                sorted_p = np.sort(node_pert)
                n_p = len(sorted_p)
                idx = np.arange(1, n_p + 1)
                gini_per_layer[l] += (
                    (2 * (idx * sorted_p).sum() / (n_p * sorted_p.sum() + 1e-12))
                    - (n_p + 1) / n_p
                )
            cls_val = float(delta_h[0].norm())
            cls_perturbation[l] += cls_val
            cls_perturbation_sq[l] += cls_val ** 2
        n_abl += 1
    except Exception:
        continue

denom = max(n_abl, 1)
mean_tv_L /= denom
max_tv_L /= denom
effect_LH /= denom
gini_per_layer /= denom
cls_perturbation /= denom
cls_perturbation_var = np.maximum(cls_perturbation_sq / denom - cls_perturbation ** 2, 0.0)
cls_perturbation_err = np.sqrt(cls_perturbation_var)
print(f'Ablation effects computed (N={n_abl})')


In [ ]:
# Fig 33–36
from graph_interp.plots.section_h import (
    fig33_ablation_tv_across_layers, fig34_ablation_effect_heatmap,
    fig35_perturbation_concentration, fig36_cls_perturbation,
)

fig = fig33_ablation_tv_across_layers(mean_tv_L, max_tv_L, N=n_abl); plt.show()
fig = fig34_ablation_effect_heatmap(effect_LH, ablate_heads=ablate_heads, N=n_abl); plt.show()
fig = fig35_perturbation_concentration(gini_per_layer, N=n_abl); plt.show()
fig = fig36_cls_perturbation(cls_perturbation, cls_perturbation_err=cls_perturbation_err, N=n_abl); plt.show()


---
## Section I: KL Divergence Scores (Figures 37–41)

In [ ]:
# ── Section I: Compute KL divergence scores ──
N_I = 10  # ← local N

from graph_interp.metrics.kl import aggregate_kl_scores
mean_kl_dot, mean_kl_bias = aggregate_kl_scores(
    model, smiles_pool, device, n_graphs=N_I,
)
print(f'KL scores: {mean_kl_dot.shape}  (N={N_I})')

In [ ]:
# Fig 37–41
from graph_interp.plots.section_i import (
    fig37_kl_dot_only_heatmap, fig38_kl_bias_only_heatmap,
    fig39_structural_dom_vs_kl_diff, fig40_kl_disagreement,
    fig41_signed_bias,
)

fig = fig37_kl_dot_only_heatmap(mean_kl_dot, N=N_I); plt.show()
fig = fig38_kl_bias_only_heatmap(mean_kl_bias, N=N_I); plt.show()
fig = fig39_structural_dom_vs_kl_diff(mean_struct, mean_sym, mean_kl_dot, mean_kl_bias, N=N_I); plt.show()
fig = fig40_kl_disagreement(mean_struct, mean_sym, mean_kl_dot, mean_kl_bias, N=N_I); plt.show()
fig = fig41_signed_bias(mean_struct, mean_sym, mean_kl_dot, mean_kl_bias, N=N_I); plt.show()

---
## Section J: Learned Bias Profiles (Figures 42–44)

In [ ]:
# ── Section J: Extract bias profiles and discover active bins ──
from graph_interp.metrics.bias_profiles import discover_active_distance_bins, bias_profile_entropy

bias_profiles, dist_labels = extract_spatial_bias_profiles(model)

# Discover which distance bins actually appear in real molecular graphs
active_bins, bin_counts, max_real_dist = discover_active_distance_bins(
    smiles_pool, model, device, max_graphs=200,
)
print(f'Bias profiles: {bias_profiles.shape}')
print(f'Active bins: {active_bins}  (max real distance: {max_real_dist})')

# Entropy/profile normalisation restricted to hop distances 1..8
entropy, entropy_norm, peak_idx, peak_d_label, profiles_prob, used_bins = bias_profile_entropy(
    bias_profiles,
    active_bins=active_bins,
    exclude_bins=[0],
    hop_min=1,
    hop_max=8,
)
print(f'Entropy computed over bins {used_bins} (hop distances 1..8 only)')


In [ ]:
# Fig 42: Bias profile entropy per head
from graph_interp.plots.section_j import fig42_bias_profile_entropy
fig = fig42_bias_profile_entropy(entropy, entropy_norm=entropy_norm)
plt.show()

In [ ]:
# Fig 43: Mode hop distance per head
from graph_interp.plots.section_j import fig43_peak_hop_distance
# Pass peak_d_label explicitly for compatibility with older cached module states in notebooks.
fig = fig43_peak_hop_distance(peak_d_label=peak_d_label, profiles_prob=profiles_prob, used_bins=used_bins)
plt.show()


In [ ]:
# Fig 44: Gallery of softmax-normalised bias profiles (active bins only)
from graph_interp.plots.section_j import fig44_bias_profiles_gallery
fig = fig44_bias_profiles_gallery(profiles_prob, used_bins)
plt.show()

---
## Section K: Entropy Diagnostics (Figures 45–48)

In [ ]:
# ── Section K: Compute entropy metrics ──
N_K = 10  # ← local N

from graph_interp.metrics.entropy import aggregate_entropy
mean_ent_bits, mean_ent_norm, mean_eff_keys, mean_cls_mass, mean_gini = aggregate_entropy(
    model, smiles_pool, device, n_graphs=N_K,
)
print(f'Entropy computed: {mean_ent_bits.shape}  (N={N_K})')

In [ ]:
# Fig 45–48
from graph_interp.plots.section_k import (
    fig45_entropy_heatmap, fig46_normalized_entropy_heatmap,
    fig47_struct_vs_entropy, fig48_symbolic_vs_entropy,
)

fig = fig45_entropy_heatmap(mean_ent_bits, N=N_K); plt.show()
fig = fig46_normalized_entropy_heatmap(mean_ent_norm, N=N_K); plt.show()
fig = fig47_struct_vs_entropy(mean_struct, mean_ent_bits, N=N_K); plt.show()
fig = fig48_symbolic_vs_entropy(mean_sym, mean_ent_bits, N=N_K); plt.show()

---
## Section L: Extra Early-Layer Specialisation — PCA Grids (Figures 49–53)

In [ ]:
# ── Section L: Compute per-head PCA embeddings for 5 layers ──
from graph_interp.chemistry import label_heads_from_attention, LABELS_FOCUS_EXPANDED

LAYERS_PCA = [0, 1, 2, 3, 11]  # ← expanded from [0, 1, 2]
K_PCA_GRID = 500               # ← 500 molecules
H = 32

FOCUS_THRESH = globals().get('FOCUS_THRESH', 0.75)
DIFFUSE_THRESH = globals().get('DIFFUSE_THRESH', 0.35)
PCA_LEGEND_NCOL = 10
PCA_LEGEND_FONTSIZE = 7.0

pca_smiles_grid = smiles_pool[:K_PCA_GRID]

# For each layer, collect [H, N_mols, Hd] embeddings and per-head labels [H, N_mols]
pca_data = {}  # layer -> (embeddings [H, N, d], label_ids [H, N])

fallback_id = LABELS_FOCUS_EXPANDED.index('other/diffuse')

for layer in LAYERS_PCA:
    emb_per_head = [[] for _ in range(H)]
    label_ids_per_head = [[] for _ in range(H)]

    for smi in pca_smiles_grid:
        try:
            batch, n, dist = build_batch_from_smiles(smi, model, device)
            _, _, A_list_tmp = extract_attention(model, batch)
            _, _, _, AV_list = extract_with_values(model, batch)

            head_labels = label_heads_from_attention(
                smi, A_list_tmp[layer],
                focus_mass=FOCUS_THRESH,
                diffuse_thresh=DIFFUSE_THRESH,
            )
            head_label_ids = [
                LABELS_FOCUS_EXPANDED.index(lab) if lab in LABELS_FOCUS_EXPANDED else fallback_id
                for lab in head_labels
            ]

            for h in range(H):
                av = AV_list[layer][h, 1:, :].mean(dim=0).cpu().numpy()
                emb_per_head[h].append(av)
                hid = head_label_ids[h] if h < len(head_label_ids) else fallback_id
                label_ids_per_head[h].append(hid)
        except Exception:
            continue

    embeddings = np.array([np.array(emb_per_head[h]) for h in range(H)])
    label_ids = np.array([np.array(label_ids_per_head[h], dtype=int) for h in range(H)], dtype=int)
    pca_data[layer] = (embeddings, label_ids)
    print(f'Layer {layer}: embeddings {embeddings.shape}, labels {label_ids.shape}')

print('All PCA embeddings computed.')


In [ ]:
# Fig 49: PCA grid — Layer 0
from graph_interp.plots.section_l import fig49_pca_layer0
emb, labs = pca_data[0]
fig = fig49_pca_layer0(
    emb, labs, N=emb.shape[1],
    focus_thresh=FOCUS_THRESH, diffuse_thresh=DIFFUSE_THRESH,
    legend_ncol=PCA_LEGEND_NCOL, legend_fontsize=PCA_LEGEND_FONTSIZE,
)
plt.show()


In [ ]:
# Fig 50: PCA grid — Layer 1
from graph_interp.plots.section_l import fig50_pca_layer1
emb, labs = pca_data[1]
fig = fig50_pca_layer1(
    emb, labs, N=emb.shape[1],
    focus_thresh=FOCUS_THRESH, diffuse_thresh=DIFFUSE_THRESH,
    legend_ncol=PCA_LEGEND_NCOL, legend_fontsize=PCA_LEGEND_FONTSIZE,
)
plt.show()


In [ ]:
# Fig 51: PCA grid — Layer 2
from graph_interp.plots.section_l import fig51_pca_layer2
emb, labs = pca_data[2]
fig = fig51_pca_layer2(
    emb, labs, N=emb.shape[1],
    focus_thresh=FOCUS_THRESH, diffuse_thresh=DIFFUSE_THRESH,
    legend_ncol=PCA_LEGEND_NCOL, legend_fontsize=PCA_LEGEND_FONTSIZE,
)
plt.show()


In [ ]:
# Fig 52: PCA grid — Layer 3 (NEW)
from graph_interp.plots.section_l import fig52_pca_layer3
emb, labs = pca_data[3]
fig = fig52_pca_layer3(
    emb, labs, N=emb.shape[1],
    focus_thresh=FOCUS_THRESH, diffuse_thresh=DIFFUSE_THRESH,
    legend_ncol=PCA_LEGEND_NCOL, legend_fontsize=PCA_LEGEND_FONTSIZE,
)
plt.show()


In [ ]:
# Fig 53: PCA grid — Layer 11 (NEW)
from graph_interp.plots.section_l import fig53_pca_layer11
emb, labs = pca_data[11]
fig = fig53_pca_layer11(
    emb, labs, N=emb.shape[1],
    focus_thresh=FOCUS_THRESH, diffuse_thresh=DIFFUSE_THRESH,
    legend_ncol=PCA_LEGEND_NCOL, legend_fontsize=PCA_LEGEND_FONTSIZE,
)
plt.show()


---

In [ ]:
# ── Section M: Compute top-k routing diagnostics ──
N_M_RESTRICTED = 30
N_M_D_SELECT = 50
N_M_ENTROPY = 100
TOP_K_M = 0.40
K_M_PERMS = 64
TAU_M = 0.02
N_M_SHUF = 10
TOP_K_FRACS_M = [0.20, 0.30, 0.40, 0.50]

from graph_interp.plots.section_m import compute_section_m_results

section_m = compute_section_m_results(
    model,
    smiles_pool,
    device,
    restricted_n_graphs=N_M_RESTRICTED,
    restricted_top_k_frac=TOP_K_M,
    restricted_num_perms=K_M_PERMS,
    restricted_tau=TAU_M,
    mass_top_k_fracs=TOP_K_FRACS_M,
    channel_n_graphs=N_M_RESTRICTED,
    channel_num_perms=K_M_PERMS,
    channel_tau=TAU_M,
    d_select_n_graphs=N_M_D_SELECT,
    d_select_n_shuf=N_M_SHUF,
    entropy_n_graphs=N_M_ENTROPY,
    verbose=True,
)

print(
    'Section M done: '
    f"restricted N={section_m['restricted']['n_used']}, "
    f"channel N={section_m['channel']['n_used']}, "
    f"d-select N={section_m['d_select']['n_used']}, "
    f"entropy N={section_m['entropy_delta']['n_used']}"
)


In [ ]:
# Fig 54–58
from graph_interp.plots.section_m import (
    fig54_partitioned_scores,
    fig55_mass_capture_multi,
    fig56_channel_attribution,
    fig57_d_selection_summary,
    fig58_entropy_delta,
)

fig = fig54_partitioned_scores(section_m['restricted']); plt.show()
fig = fig55_mass_capture_multi(
    section_m['mass_capture']['mass_capture_by_frac'],
    N=section_m['mass_capture']['n_used'],
)
plt.show()
fig = fig56_channel_attribution(section_m['channel']); plt.show()
fig = fig57_d_selection_summary(section_m['d_select']); plt.show()
fig = fig58_entropy_delta(
    section_m['entropy_delta']['delta'],
    n_used=section_m['entropy_delta']['n_used'],
)
plt.show()
